
### 03_preprocessing.ipynb - Overview


[Purpose]
- Raw JSON data -> Clean CSV for modeling
- Feature extraction from nested dictionary
- Energy consumption calculation (Target variable)

[Input]
- instance_usage-000000000000.json.gz (Google Cluster Trace)

[Output]
- instance_usage_processed.csv (cleaned data)

[Pipeline]
1. Load raw JSON (50,000 samples)
2. Extract CPU/Memory from nested 'average_usage' dict
3. Convert time (microsec -> sec)
4. Calculate duration
5. Extract hour feature (0-23)
6. Calculate power (W) and energy (kWh)
7. Save to CSV

[Energy Formula]
Power(W) = 200 + (CPU * 300) + (Memory * 50)
Energy(kWh) = Power(W) * Duration(h) / 1000

[Features Created]
- cpu: CPU usage ratio (0-1)
- memory: Memory usage ratio (0-1)
- duration: Measurement duration (seconds)
- hour: Hour of day (0-23)
- power_w: Power consumption (Watts)
- energy_kwh: Energy consumption (kWh) <- TARGET

In [ ]:
# ===========================================
# Cell 1: Google Drive Mount
# ===========================================
from google.colab import drive

drive.mount('/content/drive')
print("Drive mount done!")


In [ ]:
# ===========================================
# Cell 2: Config Load + Path Setup
# ===========================================
import os
import yaml

CONFIG_PATH = "/content/config.yaml"

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT = f"/content/drive/MyDrive/{config['project_name']}"
RAW_DATA_PATH = os.path.join(DRIVE_ROOT, config['paths']['raw_data'])
PROCESSED_DATA_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
CHECKPOINT_PATH = os.path.join(DRIVE_ROOT, "checkpoints")

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Project: {config['project_name']}")
print(f"Raw: {RAW_DATA_PATH}")
print(f"Processed: {PROCESSED_DATA_PATH}")
print(f"Checkpoints: {CHECKPOINT_PATH}")


In [ ]:
# ===========================================
# Cell 3: Library Import
# ===========================================
import pandas as pd
import numpy as np
import gzip
import json
import glob
from tqdm import tqdm
import pickle
import gc

print("Library load done!")


In [ ]:
# ===========================================
# Cell 4: List All Files
# ===========================================
all_files = sorted(glob.glob(os.path.join(RAW_DATA_PATH, "instance_usage-*.json.gz")))
print(f"Total files to process: {len(all_files)}")

# Check for checkpoint
checkpoint_file = os.path.join(CHECKPOINT_PATH, "preprocessing_checkpoint.pkl")

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'rb') as f:
        checkpoint = pickle.load(f)
    start_idx = checkpoint['last_processed_idx'] + 1
    print(f"Resuming from checkpoint: file index {start_idx}")
else:
    start_idx = 0
    print("Starting fresh (no checkpoint found)")

In [ ]:
# ===========================================
# Cell 5: Define Processing Function
# ===========================================
def process_single_file(file_path, max_rows=None):
    """
    Process a single instance_usage JSON file.

    Args:
        file_path: Path to .json.gz file
        max_rows: Limit rows per file (None = all)

    Returns:
        DataFrame with processed features
    """
    data = []

    with gzip.open(file_path, 'rt') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            data.append(json.loads(line))

    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)

    # Feature extraction
    df['cpu'] = [x.get('cpus', 0) if isinstance(x, dict) else 0 for x in df['average_usage']]
    df['memory'] = [x.get('memory', 0) if isinstance(x, dict) else 0 for x in df['average_usage']]

    # Time features
    df['start_time_sec'] = df['start_time'].astype(float) / 1000000
    df['end_time_sec'] = df['end_time'].astype(float) / 1000000
    df['duration'] = df['end_time_sec'] - df['start_time_sec']
    df['hour'] = (df['start_time_sec'] // 3600 % 24).astype(int)

    # Energy calculation
    BASE_POWER = 200
    CPU_POWER_MAX = 300
    MEM_POWER_MAX = 50

    df['power_w'] = BASE_POWER + (df['cpu'] * CPU_POWER_MAX) + (df['memory'] * MEM_POWER_MAX)
    df['energy_kwh'] = df['power_w'] * (df['duration'] / 3600) / 1000

    # Select columns
    final_columns = [
        'machine_id', 'start_time_sec', 'end_time_sec',
        'duration', 'hour', 'cpu', 'memory', 'power_w', 'energy_kwh'
    ]

    return df[final_columns]


In [ ]:
# ===========================================
# Cell 6: Process All Files (with Checkpoint)
# ===========================================
CHECKPOINT_INTERVAL = 10  # Save checkpoint every N files
SAVE_INTERVAL = 50  # Save partial results every N files

all_data = []
processed_count = 0

for idx in tqdm(range(start_idx, len(all_files)), desc="Processing files"):
    file_path = all_files[idx]

    try:
        df = process_single_file(file_path)
        all_data.append(df)
        processed_count += 1

        # Save checkpoint
        if (idx + 1) % CHECKPOINT_INTERVAL == 0:
            checkpoint = {
                'last_processed_idx': idx,
                'processed_count': processed_count
            }
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)

        # Save partial results (memory management)
        if (idx + 1) % SAVE_INTERVAL == 0:
            partial_df = pd.concat(all_data, ignore_index=True)
            partial_path = os.path.join(CHECKPOINT_PATH, f"partial_{idx+1}.parquet")
            partial_df.to_parquet(partial_path, index=False)
            print(f"\nSaved partial: {len(partial_df):,} rows to {partial_path}")

            # Clear memory
            all_data = []
            del partial_df
            gc.collect()

    except Exception as e:
        print(f"\nError processing {file_path}: {e}")
        continue

print(f"\nProcessing complete! Files processed: {processed_count}")


In [ ]:
# ===========================================
# Cell 7: Merge All Partial Files
# ===========================================
partial_files = sorted(glob.glob(os.path.join(CHECKPOINT_PATH, "partial_*.parquet")))
print(f"Found {len(partial_files)} partial files")

# Merge all partials
final_data = []

for pf in tqdm(partial_files, desc="Merging partials"):
    df = pd.read_parquet(pf)
    final_data.append(df)

# Add remaining data in memory
if all_data:
    remaining_df = pd.concat(all_data, ignore_index=True)
    final_data.append(remaining_df)

# Final merge
df_final = pd.concat(final_data, ignore_index=True)
print(f"\nTotal rows: {len(df_final):,}")


In [ ]:

# ===========================================
# Cell 8: Save Final Processed Data
# ===========================================
# Save as Parquet (faster, smaller)
output_parquet = os.path.join(PROCESSED_DATA_PATH, "instance_usage_full_processed.parquet")
df_final.to_parquet(output_parquet, index=False)
print(f"Saved: {output_parquet}")
print(f"Size: {os.path.getsize(output_parquet) / (1024**3):.2f} GB")


In [ ]:

# ===========================================
# Cell 9: Data Summary
# ===========================================
print("=" * 50)
print("Full Dataset Preprocessing Complete")
print("=" * 50)
print(f"\n[Data Statistics]")
print(f"  Total rows: {len(df_final):,}")
print(f"  Columns: {df_final.columns.tolist()}")
print(f"\n[Feature Statistics]")
print(df_final.describe())
print(f"\n[Memory Usage]")
print(f"  {df_final.memory_usage(deep=True).sum() / (1024**2):.2f} MB")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount done!
Project: EcoTracing
Raw: /content/drive/MyDrive/EcoTracing/raw
Processed: /content/drive/MyDrive/EcoTracing/processed
Checkpoints: /content/drive/MyDrive/EcoTracing/checkpoints
Library load done!
Total files to process: 5
Starting fresh (no checkpoint found)


Processing files: 100%|██████████| 5/5 [11:57<00:00, 143.55s/it]



Processing complete! Files processed: 5
Found 0 partial files


Merging partials: 0it [00:00, ?it/s]



Total rows: 19,523,808
Saved: /content/drive/MyDrive/EcoTracing/processed/instance_usage_full_processed.parquet
Size: 0.27 GB
Full Dataset Preprocessing Complete

[Data Statistics]
  Total rows: 19,523,808
  Columns: ['machine_id', 'start_time_sec', 'end_time_sec', 'duration', 'hour', 'cpu', 'memory', 'power_w', 'energy_kwh']

[Feature Statistics]
       start_time_sec  end_time_sec      duration          hour           cpu  \
count    1.952381e+07  1.952381e+07  1.952381e+07  1.952381e+07  1.952381e+07   
mean     1.376746e+06  1.376989e+06  2.433255e+02  1.133341e+01  6.747036e-03   
std      7.631083e+05  7.631123e+05  1.114370e+02  7.145479e+00  8.822266e-03   
min      3.000000e+02  6.000000e+02  1.000000e+00  0.000000e+00  0.000000e+00   
25%      7.863000e+05  7.866000e+05  3.000000e+02  5.000000e+00  3.623962e-04   
50%      1.313100e+06  1.313400e+06  3.000000e+02  1.100000e+01  5.317688e-03   
75%      2.019000e+06  2.019053e+06  3.000000e+02  1.800000e+01  8.544922e-03   
m